In [9]:
import pandas as pd
import logging
from tqdm import tqdm
import os
logger = logging.getLogger(__name__)

In [10]:
def load_data(file_path: str, year_start: int, year_end: int) -> pd.DataFrame:
    """Load and filter CSV data by year."""
    logger.info(f"Loading data for years {year_start}-{year_end}")
    try:
        df_header = pd.read_csv(file_path, nrows=0)
        expected_cols = len(df_header.columns)
        logger.info(f"Expected columns: {expected_cols}\nColumns: {df_header.columns.tolist()}")

        chunks = pd.read_csv(
            file_path,
            chunksize=1000,
            quotechar='"',
            doublequote=True,
            encoding='utf-8',
            engine='c',
            on_bad_lines='warn',
            delimiter=',',
            quoting=1
        )
        meta = []
        total_rows = 0
        estimated_rows = os.path.getsize(file_path) // 500  # rough estimate

        with tqdm(total=estimated_rows, desc="Loading data", ncols=100, colour="green") as pbar:
            for chunk in chunks:
                if len(chunk.columns) != expected_cols:
                    logger.warning(f"Found {len(chunk.columns)} columns, expected {expected_cols}")
                    continue
                chunk['year'] = pd.to_datetime(chunk['mostimportantdateutc'], errors='coerce').dt.year
                filtered_chunk = chunk[(chunk['year'] >= year_start) & (chunk['year'] <= year_end)]
                if not filtered_chunk.empty:
                    meta.append(filtered_chunk)
                    total_rows += len(filtered_chunk)
                    if total_rows % (10000) == 0:
                        logger.info(f"Processed {total_rows} rows")
                pbar.update(len(chunk))
        df_meta = pd.concat(meta, ignore_index=True)
        logger.info(f"Final dataset size: {len(df_meta)} rows")
        return df_meta

    except Exception as e:
        logger.error(f"Error in load_data: {e}")
        raise

In [11]:
file_path = "/home/zichengx/Research/AIphaBiz/poetry-demo/eCallsAgent/input_data/raw/ecc_transcripts_2006_2020.csv"
year_start = 2006
year_end = 2024

data = load_data(file_path, year_start, year_end)

2025-03-28 12:51:34,244 - __main__ - INFO - Loading data for years 2006-2024
2025-03-28 12:51:34,252 - __main__ - INFO - Expected columns: 34
Columns: ['Unnamed: 0', 'companyid', 'keydevid', 'transcriptid', 'headline', 'mostimportantdateutc', 'keydeveventtypeid', 'keydeveventtypename', 'companyname', 'transcriptcollectiontypeid', 'transcriptcollectiontypename', 'transcriptpresentationtypeid', 'transcriptpresentationtypename', 'transcriptcreationdate_utc', 'transcriptcreationtime_utc', 'audiolengthsec', 'isdelayed_flag', 'delayreasontypeid', 'delayreasontypename', 'transcriptcomponentid', 'componentorder', 'transcriptcomponenttypeid', 'transcriptpersonid', 'componenttext', 'transcriptcomponenttypename', 'transcriptpersonname', 'proid', 'companyofperson', 'speakertypeid', 'speakertypename', 'componenttextpreview', 'word_count', 'gvkey', 'year']
Loading data:  13%|████▎                             | 4315000/34225048 [02:22<13:28:18, 616.72it/s]

: 

In [6]:
from sentence_transformers import SentenceTransformer
from eCallsAgent.config.global_options import SEED_TOPICS
model = SentenceTransformer("BAAI/bge-large-en-v1.5")
seed_topic_strings = [" ".join(keywords) for keywords in SEED_TOPICS]
seed_topic_embeddings = model.encode(
    seed_topic_strings, 
    show_progress_bar=True, 
    convert_to_numpy=True,
    normalize_embeddings=True  # Ensure normalized embeddings
)
print(seed_topic_embeddings.shape)


2025-03-31 02:06:53,379 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2025-03-31 02:06:58,071 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device: cpu


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

(116, 1024)


In [9]:
embedding_matrix = [vec for vec in seed_topic_embeddings]


In [18]:
import numpy as np
np.save("/home/zichengx/Research/AIphaBiz/poetry-demo/eCallsAgent/output/seed_topic_embeddings.npy", seed_topic_embeddings)

In [13]:
print(len(seed_topic_embeddings), seed_topic_embeddings[0].shape)


116 (1024,)


In [12]:
print(len(embedding_matrix), embedding_matrix[0].shape)


116 (1024,)


In [11]:
seed_topic_embeddings

array([[-0.03464683,  0.00755031,  0.00012739, ...,  0.01759833,
        -0.01607308, -0.01508771],
       [ 0.00390746,  0.0169139 , -0.01413361, ..., -0.04784415,
        -0.0118276 , -0.02289276],
       [-0.03563976,  0.01556434, -0.0378576 , ..., -0.00356862,
        -0.03635583,  0.02582046],
       ...,
       [-0.02265683, -0.00727033, -0.002517  , ...,  0.00787586,
        -0.01385609,  0.02010448],
       [-0.03341407,  0.05146318,  0.00356167, ..., -0.02692966,
        -0.00772836, -0.01907352],
       [-0.03449149,  0.01331955, -0.00559633, ...,  0.01694266,
        -0.04449249,  0.00465775]], dtype=float32)

In [17]:
assert isinstance(embedding_matrix, np.ndarray)
assert embedding_matrix.shape[1] == 1024
assert len(SEED_TOPICS) == embedding_matrix.shape[0]

AssertionError: 

In [1]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))["data"]

seed_topic_list = [["drug", "cancer", "drugs", "doctor"],
                   ["windows", "drive", "dos", "file"],
                   ["space", "launch", "orbit", "lunar"]]

topic_model = BERTopic(seed_topic_list=seed_topic_list)
topics, probs = topic_model.fit_transform(docs)

KeyboardInterrupt: 